# Project 1: Thermal Sensation — Day 2: EDA and feature engineering

Today we will look for patterns in the cleaned thermal-sensation data and decide which columns are reasonable model inputs.

To keep our final evaluation fair, we first set aside some whole buildings. We will use the training buildings for EDA and feature choices. We will save the test buildings for final model evaluation, not use their results to make modeling decisions.

## 1. Load the cleaned data and tools

This file is created by the Day 1 notebook. It already has building metadata joined and rows without a thermal-sensation target removed. Missing input features are still present; we will decide how to handle those later in the model pipeline.

The joined table has columns with different data types. `low_memory=False` asks pandas to inspect the full CSV when choosing types, which avoids mixed-type warnings while loading.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.model_selection import GroupShuffleSplit

# Find the project folder whether this notebook is opened from the root or notebooks/.
project_folder = Path.cwd()
if not (project_folder / "data" / "processed").exists():
    project_folder = project_folder.parent

cleaned_data_path = (
    project_folder / "data" / "processed" / "thermal_sensation_cleaned.csv"
)

if not cleaned_data_path.exists():
    raise FileNotFoundError(
        "Run the Day 1 notebook first. Cleaned data was not found at: "
        f"{cleaned_data_path}"
    )

cleaned_data = pd.read_csv(
    cleaned_data_path,
    dtype={"building_id": "string", "subject_id": "string"},
    low_memory=False,
)

print("Cleaned data:", cleaned_data.shape[0], "rows and", cleaned_data.shape[1], "columns")

## 2. Make and save a building-based training and test split

Many rows can come from the same building. If rows from one building appear in both training and test data, the evaluation may be too optimistic. `GroupShuffleSplit` keeps each `building_id` entirely in one group.

We will use about 20% of buildings for the test set. The test data should not be used to choose features or tune a model. We will save the split assignment so the later notebooks use exactly the same buildings.

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)

# The splitter returns row positions for train and test groups.
train_positions, test_positions = next(
    splitter.split(cleaned_data, groups=cleaned_data["building_id"])
)

training_data = cleaned_data.iloc[train_positions].copy()
test_data = cleaned_data.iloc[test_positions].copy()

training_buildings = set(training_data["building_id"].unique())
test_buildings = set(test_data["building_id"].unique())

print("Training rows:", len(training_data))
print("Test rows:", len(test_data))
print("Training buildings:", len(training_buildings))
print("Test buildings:", len(test_buildings))
print("Buildings shared between groups:", len(training_buildings & test_buildings))

assert training_buildings.isdisjoint(test_buildings)

# Save only the building IDs and split names so later notebooks can repeat this split.
split_assignments = pd.concat(
    [
        pd.DataFrame({"building_id": sorted(training_buildings), "data_split": "train"}),
        pd.DataFrame({"building_id": sorted(test_buildings), "data_split": "test"}),
    ],
    ignore_index=True,
)

split_path = project_folder / "data" / "processed" / "p1_building_split.csv"
split_assignments.to_csv(split_path, index=False)
print("Building split saved to:", split_path)

## 3. Explore the training data

First, check the target distribution. The sensation values include decimals, so a histogram is more informative than treating every distinct decimal number as a separate class.

In [ ]:
print("Thermal sensation summary in training data:")
display(training_data["thermal_sensation"].describe())

plt.figure(figsize=(8, 5))
sns.histplot(data=training_data, x="thermal_sensation", bins=30)
plt.title("Thermal sensation votes in the training data")
plt.xlabel("Thermal sensation (-3 = cold, +3 = hot)")
plt.ylabel("Number of observations")
plt.show()

Now look at how complete the possible inputs are. A column with many missing values may still be useful, but we need to know how much information it contributes and how many records it affects. We will not fill in missing values during EDA.

In [ ]:
candidate_numeric_features = ["ta", "top", "tr", "rh", "vel", "clo", "met"]

feature_missing_percent = (
    training_data[candidate_numeric_features].isna().mean() * 100
).sort_values(ascending=False)

print("Missing percentage for candidate numeric features:")
display(feature_missing_percent.round(1).to_frame(name="missing_percent"))

print("Numeric feature summaries:")
display(training_data[candidate_numeric_features].describe().T)

In [ ]:
# Plot the distributions of the main indoor measurements.
training_data[candidate_numeric_features].hist(figsize=(12, 9), bins=30)
plt.suptitle("Candidate measurement distributions in training data")
plt.tight_layout()
plt.show()

In [ ]:
# Show the relationship between air temperature and the target.
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=training_data,
    x="ta",
    y="thermal_sensation",
    alpha=0.15,
    s=15,
)
plt.title("Air temperature and reported thermal sensation")
plt.xlabel("Air temperature (°C)")
plt.ylabel("Thermal sensation")
plt.show()

## 4. Choose a first set of features

For a first model, prefer measurements that are reasonably complete and have a clear physical meaning. `ta`, `rh`, `vel`, `clo`, and `met` are good initial candidates. `top` and `tr` can be studied as optional features, but their missingness is higher.

We can include global context such as season, country, climate, building type, and cooling type. We will not use IDs as predictors because they can let a model memorize a particular building or study.

In [ ]:
numeric_features = ["ta", "rh", "vel", "clo", "met"]
categorical_features = [
    "season",
    "country",
    "climate",
    "building_type",
    "cooling_type",
]
target_column = "thermal_sensation"

selected_features = numeric_features + categorical_features

print("Target:", target_column)
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

# Check that every planned column is present before moving on.
missing_columns = [
    column
    for column in selected_features + [target_column]
    if column not in training_data.columns
]
print("Planned columns missing from the data:", missing_columns)
assert not missing_columns

### Why some columns are not model inputs

- We do not use `pmv`, `pmv_ce`, `ppd`, `ppd_ce`, or `set`. These are calculated comfort indices and could make the model appear to predict occupant sensation while mostly reproducing another formula.
- We do not use `thermal_preference`, `thermal_acceptability`, or other occupant votes to predict `thermal_sensation`. Those are separate responses, not environmental inputs.
- We do not use `record_id`, `building_id`, `subject_id`, or publication as ordinary predictors. Building IDs will instead help keep the training and test groups separate.
- We will not drop rows for missing input features here. Day 3 will place missing-value handling inside the model pipeline so it learns from training data only.

The feature list above is an initial, explainable baseline. Later experiments can compare alternatives without changing the held-out test data.